In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler,normalize
from sklearn.decomposition import IncrementalPCA
import h5py

import gc

In [ ]:
data = np.load("user_means_and_counts.npz", allow_pickle=True)

user_means = data["user_means"].item()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/user_means_and_counts.npz'

In [ ]:
ipca= IncrementalPCA(n_components=100, batch_size=100_000)

In [ ]:
names = list(user_means.keys())
batch_size = ipca.batch_size

In [ ]:
for i in range(0, len(names), batch_size):
    batch_names = names[i:i+batch_size]

    # 2D-Array für IPCA
    X_batch = np.stack([user_means[n] for n in batch_names]).astype(np.float32)

    ipca.partial_fit(X_batch)

    del X_batch

In [ ]:
import joblib

joblib.dump(ipca, "pca_model.joblib")

['/content/drive/MyDrive/pca_model.joblib']

In [ ]:
for i in range(0, len(names), batch_size):
    batch_names = names[i:i+batch_size]

    # Batch bauen
    X_batch = np.stack([user_means[n] for n in batch_names]).astype(np.float32)

    # PCA transform
    X_batch_pca = ipca.transform(X_batch)

    # Inplace überschreiben
    for j, name in enumerate(batch_names):
        user_means[name] = X_batch_pca[j]

    del X_batch, X_batch_pca

In [ ]:
user_counts = data["user_counts"].item()

In [ ]:
np.savez_compressed(
    "user_pca_reduced_and_counts.npz",
    user_pca_reduced=user_means,
    user_counts=user_counts
)